# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets and fields using their `@id`s.

**Note:** In Croissant, record sets and fields are referenced via unique `@id` URIs.

Let's list available record sets and their fields:

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets
print("Available record sets and their field IDs:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    fields = rs.get('field', [])
    if not fields:
        print("  No fields found.")
        continue
    for field in fields:
        if isinstance(field, dict):  # direct field dict
            fid = field.get('@id', 'Unknown')
            label = field.get('label', '')
            print(f"  Field @id: {fid} | Label: {label}")
        else:
            print(f"  Field @id: {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we will extract records from each record set, referencing entities by their `@id`.

In [ ]:
# Extract data from record sets
rs_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in rs_ids:
    print(f"\nLoading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. For illustration, let's use the first record set and select a numeric field (for example, GAD-7 score). All references will use entity `@id`s.

In [ ]:
# Choose the main record set for analysis (the first RecordSet as example)
main_record_set_id = rs_ids[0]
df = dataframes[main_record_set_id]

# Display column names and select a numeric field by @id
print(f"Available columns (@id): {df.columns.tolist()}")

# Assume GAD-7 score is represented by its Croissant field @id (replace as necessary)
# Let's search for a GAD-7 column (e.g., 'gad7_score' or similar)
import re
gad7_field_id = None
for col in df.columns:
    if re.search(r'gad.?7', col, re.IGNORECASE):
        gad7_field_id = col
        break
print(f"Selected numeric field (GAD-7): {gad7_field_id}")

# Set threshold for filtering
threshold = 10
filtered_df = df[df[gad7_field_id] > threshold]
print(f"Filtered records with {gad7_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print("Normalized GAD-7 scores:")
print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Grouping by another field (e.g., 'level_of_education' - replace with actual @id)
edu_field_id = None
for col in df.columns:
    if re.search(r'education', col, re.IGNORECASE):
        edu_field_id = col
        break
print(f"Group field candidate: {edu_field_id}")
if edu_field_id:
    grouped_df = filtered_df.groupby(edu_field_id)[gad7_field_id].mean().reset_index()
    print(f"Mean {gad7_field_id} by {edu_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of GAD-7 scores and the relationship with level of education (if available).

We use `matplotlib` and `seaborn` for basic visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
plt.figure(figsize=(8, 4))
sns.histplot(df[gad7_field_id], bins=15, kde=True)
plt.title('Distribution of GAD-7 Scores')
plt.xlabel('GAD-7 Score')
plt.ylabel('Frequency')
plt.show()

# Boxplot of GAD-7 by Level of Education
if edu_field_id:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[edu_field_id], y=df[gad7_field_id])
    plt.title('GAD-7 Scores by Level of Education')
    plt.xlabel('Level of Education')
    plt.ylabel('GAD-7 Score')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and analyze a Croissant dataset using the `mlcroissant` library.

Key observations:
- Dataset contains demographic and mental health indicators from Kilifi County, Kenya.
- You can filter, normalize, and group by any fields using their `@id`.
- Visualizations facilitate understanding of score distributions and possible demographic influences.

For further analysis, refer to specific field `@id`s and documentation for more advanced processing.